# 🟩 좌표압축 (Coordinate Compression) 완전 가이드  

## 📖 목차  
1. [개념 소개](#-개념-소개)  
2. [필요성과 활용](#-필요성과-활용)  
3. [구현 방법](#-구현-방법)  
4. [상세 예제](#-상세-예제)  
5. [응용 문제들](#-응용-문제들)  
6. [시간복잡도 분석](#-시간복잡도-분석)  
7. [주의사항과 팁](#-주의사항과-팁)  

---

## 🎯 개념 소개  

**좌표압축(Coordinate Compression)**은 큰 범위의 좌표값들을 작은 범위로 압축하여 메모리와 시간을 절약하는 알고리즘 기법입니다.  

### 핵심 아이디어  
- 실제 좌표값의 **상대적 순서**만 중요한 경우에 사용  
- 큰 좌표값들을 0, 1, 2, ... 와 같은 작은 값으로 변환  
- **이산화(Discretization)**의 한 종류  

### 기본 원리  
```
원본 좌표: [1000000, 5, 999999, 100]  
정렬 후:  [5, 100, 999999, 1000000]  
압축 결과: 5→0, 100→1, 999999→2, 1000000→3  
```

---

## 💡 필요성과 활용  

### 언제 사용하나요?  

#### ✅ 사용하는 경우  
- 좌표값이 매우 크지만 개수는 적을 때 (예: -10⁹ ≤ x ≤ 10⁹, N ≤ 1000)  
- 배열 인덱스로 좌표를 사용해야 할 때  
- 세그먼트 트리, 펜윅 트리 등에서 큰 좌표 범위를 다룰 때  
- 2차원 평면에서 점들의 상대적 위치만 중요할 때  

#### ❌ 사용하지 않는 경우  
- 좌표값 자체가 중요한 계산에서  
- 거리나 면적 계산이 필요한 경우  
- 연속적인 범위 처리가 필요한 경우  

### 대표적인 활용 분야  
- **구간 쿼리 문제**: 좌표가 큰 구간에서의 합, 최댓값 등  
- **기하 문제**: 점들의 상대적 위치 비교  
- **압축된 DP**: 상태 공간이 클 때  
- **이벤트 처리**: 시간이나 위치 기반 이벤트  

---

## 🔧 구현 방법  

### 1차원 좌표압축  

```cpp
#include <algorithm>  
#include <vector>  
using namespace std;  

vector<int> compress_coordinates(vector<int>& coords) {  
    // 1. 중복 제거를 위한 복사본 생성  
    vector<int> sorted_coords = coords;  
    
    // 2. 정렬  
    sort(sorted_coords.begin(), sorted_coords.end());  
    
    // 3. 중복 제거  
    sorted_coords.erase(unique(sorted_coords.begin(), sorted_coords.end()),  
                       sorted_coords.end());  
    
    // 4. 압축된 좌표로 변환  
    vector<int> compressed;  
    for (int coord : coords) {  
        int compressed_value = lower_bound(sorted_coords.begin(),  
                                         sorted_coords.end(), coord)  
                              - sorted_coords.begin();  
        compressed.push_back(compressed_value);  
    }  
    
    return compressed;  
}  
```

### 2차원 좌표압축  

```cpp
pair<vector<int>, vector<int>> compress_2d(vector<pair<int, int>>& points) {  
    vector<int> x_coords, y_coords;  
    
    // 좌표 분리  
    for (auto& point : points) {  
        x_coords.push_back(point.first);  
        y_coords.push_back(point.second);  
    }  
    
    // 각각 압축  
    vector<int> compressed_x = compress_coordinates(x_coords);  
    vector<int> compressed_y = compress_coordinates(y_coords);  
    
    return {compressed_x, compressed_y};  
}  
```

---

## 📚 상세 예제  

### 예제 1: 기본 1차원 압축  

```
입력: [1000000, 5, 999999, 100, 5]  
```

**단계별 처리:**  

1. **정렬 및 중복 제거**  
   ```
   정렬: [5, 100, 999999, 1000000]  
   ```

2. **매핑 테이블 생성**  
   ```
   5       → 0  
   100     → 1  
   999999  → 2  
   1000000 → 3  
   ```

3. **결과**  
   ```
   원본:     [1000000, 5, 999999, 100, 5]  
   압축결과: [3, 0, 2, 1, 0]  
   ```

### 예제 2: 구간 쿼리 문제  

**문제**: N개의 구간 [L, R]이 주어졌을 때, 각 점이 몇 개의 구간에 포함되는지 구하기  

```cpp
#include <vector>  
#include <algorithm>  
using namespace std;  

vector<int> count_intervals(vector<pair<int, int>>& intervals) {  
    vector<int> coords;  
    
    // 모든 구간의 끝점 수집  
    for (auto& interval : intervals) {  
        coords.push_back(interval.first);   // 시작점  
        coords.push_back(interval.second);  // 끝점  
    }  
    
    // 좌표 압축  
    sort(coords.begin(), coords.end());  
    coords.erase(unique(coords.begin(), coords.end()), coords.end());  
    
    // 차분 배열 생성  
    vector<int> diff(coords.size() + 1, 0);  
    
    for (auto& interval : intervals) {  
        int left = lower_bound(coords.begin(), coords.end(), interval.first)  
                  - coords.begin();  
        int right = lower_bound(coords.begin(), coords.end(), interval.second)  
                   - coords.begin();  
        
        diff[left]++;  
        diff[right + 1]--;  
    }  
    
    // 누적합으로 실제 카운트 계산  
    vector<int> count(coords.size());  
    count[0] = diff[0];  
    for (int i = 1; i < coords.size(); i++) {  
        count[i] = count[i-1] + diff[i];  
    }  
    
    return count;  
}  
```

### 예제 3: 2차원 격자 문제  

**문제**: 2차원 평면의 점들을 격자로 압축하여 처리  

```cpp
class CompressedGrid {  
private:  
    vector<int> x_vals, y_vals;  
    vector<vector<int>> grid;  
    
public:  
    CompressedGrid(vector<pair<int, int>>& points) {  
        // x, y 좌표 분리 및 수집  
        for (auto& p : points) {  
            x_vals.push_back(p.first);  
            y_vals.push_back(p.second);  
        }  
        
        // 정렬 및 중복 제거  
        sort(x_vals.begin(), x_vals.end());  
        sort(y_vals.begin(), y_vals.end());  
        x_vals.erase(unique(x_vals.begin(), x_vals.end()), x_vals.end());  
        y_vals.erase(unique(y_vals.begin(), y_vals.end()), y_vals.end());  
        
        // 격자 초기화  
        grid.resize(x_vals.size(), vector<int>(y_vals.size(), 0));  
    }  
    
    void mark_point(int x, int y) {  
        int compressed_x = lower_bound(x_vals.begin(), x_vals.end(), x)  
                          - x_vals.begin();  
        int compressed_y = lower_bound(y_vals.begin(), y_vals.end(), y)  
                          - y_vals.begin();  
        grid[compressed_x][compressed_y] = 1;  
    }  
    
    int query_rectangle(int x1, int y1, int x2, int y2) {  
        int cx1 = lower_bound(x_vals.begin(), x_vals.end(), x1) - x_vals.begin();  
        int cy1 = lower_bound(y_vals.begin(), y_vals.end(), y1) - y_vals.begin();  
        int cx2 = upper_bound(x_vals.begin(), x_vals.end(), x2) - x_vals.begin() - 1;  
        int cy2 = upper_bound(y_vals.begin(), y_vals.end(), y2) - y_vals.begin() - 1;  
        
        int count = 0;  
        for (int i = cx1; i <= cx2; i++) {  
            for (int j = cy1; j <= cy2; j++) {  
                count += grid[i][j];  
            }  
        }  
        return count;  
    }  
};  
```

---

## 🎮 응용 문제들  

### 1. 백준 문제 예시  

- **[좌표 압축 (18870)](https://www.acmicpc.net/problem/18870)**: 기본 좌표압축 문제  
- **[가장 긴 증가하는 부분 수열 2 (12015)](https://www.acmicpc.net/problem/12015)**: LIS + 좌표압축  
- **[구간 합 구하기 3 (2042)](https://www.acmicpc.net/problem/2042)**: 세그먼트 트리 + 좌표압축  

### 2. 문제 유형별 접근법  

#### 📊 구간/범위 문제  
```cpp
// 스위핑 + 좌표압축  
vector<int> events;  // 이벤트 지점들  
// 압축 후 세그먼트 트리나 펜윅 트리 사용  
```

#### 🎯 기하 문제  
```cpp
// 점들의 상대적 위치만 중요한 경우  
vector<pair<int, int>> points;  
auto [compressed_x, compressed_y] = compress_2d(points);  
```

#### 💫 동적 프로그래밍  
```cpp
// 상태 공간이 큰 DP  
map<int, int> compress_map;  // 원본 → 압축  
vector<int> dp(compressed_size);  
```

---

## ⏱️ 시간복잡도 분석  

### 복잡도 비교  

| 연산 | 압축 전 | 압축 후 |  
|------|---------|---------|
| **전처리** | O(1) | O(N log N) |  
| **메모리** | O(max_coord) | O(N) |  
| **쿼리** | O(1) | O(log N) |  
| **공간 효율** | 매우 낮음 | 높음 |  

### 상세 분석  

```
N: 좌표의 개수  
M: 좌표의 최댓값  

압축 과정:  
- 정렬: O(N log N)  
- 중복 제거: O(N)  
- 이진 탐색: O(N log N)  
총 시간복잡도: O(N log N)  

공간복잡도:  
- 압축 전: O(M) - 최대 10^9까지 가능  
- 압축 후: O(N) - 실제 사용하는 좌표만  
```

---

## ⚠️ 주의사항과 팁  

### 🚨 흔한 실수들  

#### 1. 중복 처리 실수  
```cpp
// ❌ 잘못된 예  
sort(coords.begin(), coords.end());  
// unique() 호출 없이 바로 사용  

// ✅ 올바른 예  
sort(coords.begin(), coords.end());  
coords.erase(unique(coords.begin(), coords.end()), coords.end());  
```

#### 2. 경계 처리 오류  
```cpp
// ❌ 잘못된 예 - 구간 끝점이 포함되지 않을 수 있음  
int right = lower_bound(...) - coords.begin();  

// ✅ 올바른 예 - 구간의 끝점 포함 여부 확인  
int right = upper_bound(...) - coords.begin() - 1;  
```

#### 3. 원본 좌표 필요시 역변환 누락  
```cpp
// 압축된 인덱스 → 원본 좌표 변환  
int original_coord = coords[compressed_index];  
```

### 💡 최적화 팁  

#### 1. `map` vs `vector` + 이진탐색  
```cpp
// 메모리와 속도 모두 고려하면 vector가 유리  
map<int, int> coord_map;  // 느림  
vector<int> coords + lower_bound();  // 빠름  
```

#### 2. 메모리 사용량 최소화  
```cpp
// 압축 후 원본 배열 해제  
vector<int>().swap(original_coords);  
```

#### 3. 구간 처리시 오프셋 활용  
```cpp
// 구간 [L, R]에서 R+1도 필요한 경우가 많음  
coords.push_back(interval.second + 1);  
```

### 🎯 디버깅 팁  

```cpp
void debug_compression(vector<int>& original, vector<int>& compressed,  
                      vector<int>& coord_table) {  
    cout << "Original → Compressed:\n";  
    for (int i = 0; i < original.size(); i++) {  
        cout << original[i] << " → " << compressed[i] << "\n";  
    }  
    
    cout << "\nCompression table:\n";  
    for (int i = 0; i < coord_table.size(); i++) {  
        cout << i << " : " << coord_table[i] << "\n";  
    }  
}  
```

---

## 🎉 마무리  

좌표압축은 큰 좌표 범위를 작은 범위로 효율적으로 변환하는 강력한 기법입니다. 특히 알고리즘 대회나 실무에서 메모리 제한이 있는 상황에서 매우 유용합니다.  

**핵심 포인트:**  
- 상대적 순서만 중요할 때 사용  
- 정렬 → 중복제거 → 매핑의 3단계 과정  
- `lower_bound()`와 `upper_bound()` 활용  
- 원본 좌표 복원 가능성 고려  

**다음 학습 권장사항:**  
- 세그먼트 트리와 조합한 고급 문제  
- 2차원 좌표압축 응용  
- 오프라인 쿼리 처리 기법  